# RNNs - Practice Exercise

Experiment with sequence models:
1. Compare LSTM vs GRU architectures
2. Test with different embedding dimensions
3. Analyze sentiment predictions
4. Experiment with bidirectional models

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import sequence

# Load IMDB
VOCAB_SIZE, MAX_LEN = 10000, 200
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
X_train = sequence.pad_sequences(X_train, maxlen=MAX_LEN)
X_test = sequence.pad_sequences(X_test, maxlen=MAX_LEN)

print(f"Data shape: {X_train.shape}")

## Exercise: LSTM vs GRU vs Bidirectional

Compare different RNN configurations on sentiment analysis.

In [ ]:
# Model 1: LSTM
model_lstm = tf.keras.Sequential([
    tf.keras.layers.Embedding(VOCAB_SIZE, 128, input_length=MAX_LEN),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Model 2: GRU (simpler than LSTM)
model_gru = tf.keras.Sequential([
    tf.keras.layers.Embedding(VOCAB_SIZE, 128, input_length=MAX_LEN),
    tf.keras.layers.GRU(64),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Model 3: Bidirectional LSTM (reads both directions)
model_bidir = tf.keras.Sequential([
    tf.keras.layers.Embedding(VOCAB_SIZE, 128, input_length=MAX_LEN),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

configs = {'LSTM': model_lstm, 'GRU': model_gru, 'Bidirectional': model_bidir}
results = {}

for name, model in configs.items():
    print(f"\nTraining {name}...")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    hist = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    results[name] = {'history': hist, 'test_acc': test_acc, 'params': model.count_params()}
    print(f"{name}: Test Acc = {test_acc:.4f}, Params = {model.count_params():,}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name in configs.keys():
    hist = results[name]['history']
    axes[0].plot(hist.history['val_accuracy'], label=name, marker='o', markersize=4)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('RNN Architecture Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

names = list(results.keys())
test_accs = [results[n]['test_acc'] for n in names]
axes[1].bar(names, test_accs, alpha=0.7)
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_ylim([0.80, 0.90])
for i, v in enumerate(test_accs):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=8, fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nQuestion: Does bidirectionality help? How do LSTM and GRU compare?")